In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import os

In [2]:
cols_fecha = ['year', 'month', 'day']
cols_elo = ['elo_h', 'elo_a']
#quito ruido pasandole solo los deltas
cols_avanzadas = [
    'expected_pace', 
    'efg_diff', 
    'tov_diff', 
    'oreb_diff', 
    'ft_rate_diff', 
    'rest_diff'
]
cols_equipos = []

In [3]:
# --- 1. FUNCIÓN PARA EL RITMO (PACE) ---
def procesar_ritmo_play_by_play(ruta_pbp, ruta_games):
    print("Procesando Ritmo (Pace) desde Play-by-Play...")
    columnas_pbp = ['game_id', 'eventmsgtype', 'player1_team_id']
    pbp = pd.read_csv(ruta_pbp, usecols=columnas_pbp)
    
    eventos_posesion = pbp[pbp['eventmsgtype'].isin([1, 2, 3, 5])]
    stats = eventos_posesion.groupby(['game_id', 'player1_team_id', 'eventmsgtype']).size().unstack(fill_value=0).reset_index()
    
    for col in [1, 2, 3, 5]:
        if col not in stats.columns:
            stats[col] = 0
            
    stats['possessions'] = stats[1] + stats[2] + (0.44 * stats[3]) + stats[5]
    df_pace = stats[['game_id', 'player1_team_id', 'possessions']].copy()
    df_pace.columns = ['game_id', 'team_id', 'pace']
    
    games = pd.read_csv(ruta_games, usecols=['game_id', 'game_date'])
    games['game_date'] = pd.to_datetime(games['game_date'])
    df_pace = pd.merge(df_pace, games, on='game_id', how='inner')
    
    return df_pace[['game_date', 'team_id', 'pace']]


In [4]:
# --- 2. FUNCIÓN PARA LOS FOUR FACTORS ---
def calcular_four_factors(ruta_games):
    print("Calculando los Four Factors de Dean Oliver...")
    cols = ['game_date', 'team_id_home', 'team_id_away',
            'fgm_home', 'fga_home', 'fg3m_home', 'tov_home', 'fta_home', 'ftm_home', 'oreb_home', 'dreb_home',
            'fgm_away', 'fga_away', 'fg3m_away', 'tov_away', 'fta_away', 'ftm_away', 'oreb_away', 'dreb_away']
    
    games = pd.read_csv(ruta_games, usecols=cols).dropna()
    games['game_date'] = pd.to_datetime(games['game_date'])

    games['efg_home'] = (games['fgm_home'] + 0.5 * games['fg3m_home']) / games['fga_home']
    games['tov_pct_home'] = games['tov_home'] / (games['fga_home'] + 0.44 * games['fta_home'] + games['tov_home'])
    games['oreb_pct_home'] = games['oreb_home'] / (games['oreb_home'] + games['dreb_away'])
    games['ft_rate_home'] = games['ftm_home'] / games['fga_home']

    games['efg_away'] = (games['fgm_away'] + 0.5 * games['fg3m_away']) / games['fga_away']
    games['tov_pct_away'] = games['tov_away'] / (games['fga_away'] + 0.44 * games['fta_away'] + games['tov_away'])
    games['oreb_pct_away'] = games['oreb_away'] / (games['oreb_away'] + games['dreb_home'])
    games['ft_rate_away'] = games['ftm_away'] / games['fga_away']

    home_ff = games[['game_date', 'team_id_home', 'efg_home', 'tov_pct_home', 'oreb_pct_home', 'ft_rate_home']].copy()
    home_ff.columns = ['game_date', 'team_id', 'efg', 'tov_pct', 'oreb_pct', 'ft_rate']

    away_ff = games[['game_date', 'team_id_away', 'efg_away', 'tov_pct_away', 'oreb_pct_away', 'ft_rate_away']].copy()
    away_ff.columns = ['game_date', 'team_id', 'efg', 'tov_pct', 'oreb_pct', 'ft_rate']

    team_ff = pd.concat([home_ff, away_ff]).sort_values(['team_id', 'game_date']).reset_index(drop=True)

    team_ff['avg_efg_5'] = team_ff.groupby('team_id')['efg'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.50)
    team_ff['avg_tov_pct_5'] = team_ff.groupby('team_id')['tov_pct'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.15)
    team_ff['avg_oreb_pct_5'] = team_ff.groupby('team_id')['oreb_pct'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.25)
    team_ff['avg_ft_rate_5'] = team_ff.groupby('team_id')['ft_rate'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(0.20)

    return team_ff[['game_date', 'team_id', 'avg_efg_5', 'avg_tov_pct_5', 'avg_oreb_pct_5', 'avg_ft_rate_5']]


In [5]:
# --- 3. ONE-HOT ENCODING ---
def preparar_datos_ohe(df, cols_equipos):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy['year'] = df_copy['game_date'].dt.year
    df_copy['month'] = df_copy['game_date'].dt.month
    df_copy['day'] = df_copy['game_date'].dt.day

    df_copy = pd.get_dummies(df_copy, columns=['team_abbreviation_home','team_abbreviation_away'], dtype=float)
    if cols_equipos == []:
        cols_equipos = [c for c in df_copy.columns if 'team_abbreviation_home_' in c or 'team_abbreviation_away_' in c]
    return df_copy, cols_equipos


In [6]:
# --- 4. INTEGRACIÓN DE MATCHUP DELTAS ---
def generar_features_avanzadas(df, df_pace, df_four_factors):
    df_copy = df.copy()
    df_copy['game_date'] = pd.to_datetime(df_copy['game_date'])
    df_copy = df_copy.sort_values('game_date').reset_index(drop=True)

    home_df = df_copy[['game_date', 'team_id_home', 'pts_home', 'pts_away']].rename(
        columns={'team_id_home': 'team_id', 'pts_home': 'pts_scored', 'pts_away': 'pts_allowed'})
    home_df['is_home'] = 1

    away_df = df_copy[['game_date', 'team_id_away', 'pts_away', 'pts_home']].rename(
        columns={'team_id_away': 'team_id', 'pts_away': 'pts_scored', 'pts_home': 'pts_allowed'})
    away_df['is_home'] = 0

    team_games = pd.concat([home_df, away_df]).sort_values(['team_id', 'game_date']).reset_index(drop=True)

    # Añadir Ritmo y Four Factors
    team_games = pd.merge(team_games, df_pace, on=['game_date', 'team_id'], how='left')
    team_games['pace'] = team_games['pace'].fillna(100)
    team_games = pd.merge(team_games, df_four_factors, on=['game_date', 'team_id'], how='left')

    # Días de descanso continuos
    team_games['rest_days'] = team_games.groupby('team_id')['game_date'].diff().dt.days
    team_games['rest_days'] = team_games['rest_days'].fillna(14).clip(upper=14)

    # Medias Móviles
    team_games['avg_pace_5'] = team_games.groupby('team_id')['pace'].transform(lambda x: x.rolling(5).mean().shift(1)).fillna(100)
    
    # Separar Locales
    cols_home = ['game_date', 'team_id', 'rest_days', 'avg_pace_5', 'avg_efg_5', 'avg_tov_pct_5', 'avg_oreb_pct_5', 'avg_ft_rate_5']
    home_features = team_games[team_games['is_home'] == 1][cols_home]
    home_features.columns = ['game_date', 'team_id_home', 'rest_days_home', 'avg_pace_home', 'avg_efg_home', 'avg_tov_pct_home', 'avg_oreb_pct_home', 'avg_ft_rate_home']

    # Separar Visitantes
    cols_away = ['game_date', 'team_id', 'rest_days', 'avg_pace_5', 'avg_efg_5', 'avg_tov_pct_5', 'avg_oreb_pct_5', 'avg_ft_rate_5']
    away_features = team_games[team_games['is_home'] == 0][cols_away]
    away_features.columns = ['game_date', 'team_id_away', 'rest_days_away', 'avg_pace_away', 'avg_efg_away', 'avg_tov_pct_away', 'avg_oreb_pct_away', 'avg_ft_rate_away']

    # Fusionar con dataset original
    df_copy = pd.merge(df_copy, home_features, on=['game_date', 'team_id_home'], how='left')
    df_copy = pd.merge(df_copy, away_features, on=['game_date', 'team_id_away'], how='left')
    df_copy = df_copy.drop_duplicates(subset=['team_id_home', 'team_id_away', 'game_date'])

    # ----------------------------------------------------------------
    # CREACIÓN DE LAS VARIABLES DELTA (El núcleo de este experimento)
    # ----------------------------------------------------------------
    # 1. Ritmo esperado del partido (Media de los ritmos de ambos equipos)
    df_copy['expected_pace'] = (df_copy['avg_pace_home'] + df_copy['avg_pace_away']) / 2.0
    
    # 2. Batallas de Eficiencia (Local - Visitante)
    df_copy['efg_diff'] = df_copy['avg_efg_home'] - df_copy['avg_efg_away']
    df_copy['tov_diff'] = df_copy['avg_tov_pct_home'] - df_copy['avg_tov_pct_away']
    df_copy['oreb_diff'] = df_copy['avg_oreb_pct_home'] - df_copy['avg_oreb_pct_away']
    df_copy['ft_rate_diff'] = df_copy['avg_ft_rate_home'] - df_copy['avg_ft_rate_away']
    
    # 3. Diferencia de descanso
    df_copy['rest_diff'] = df_copy['rest_days_home'] - df_copy['rest_days_away']

    return df_copy

In [7]:
def escalar_datos(df, df_test, cols_no_escalables, cols_escalables):
    scaler = StandardScaler()
    scaler.fit(df[cols_escalables])
    df_escalado = scaler.transform(df[cols_escalables])
    df_test_escalado = scaler.transform(df_test[cols_escalables])
    data_entrada = np.hstack([np.array(df[cols_no_escalables]), df_escalado])
    data_entrada_test = np.hstack([np.array(df_test[cols_no_escalables]), df_test_escalado])
    return data_entrada, data_entrada_test

def preparar_datos_salida(df):
    return np.column_stack((df['pts_home'].values, df['pts_away'].values))


In [8]:
# ----------------- EJECUCIÓN -----------------
print("Cargando y cruzando bases de datos...")
df_partidos_elo1 = pd.read_csv('csv_red/partidos_elo1.csv')
RUTA_GAME = "../../data/inputs/csv/game.csv" 
RUTA_PBP = "../../data/inputs/csv/play_by_play.csv"

# Calculamos las dos mega-tablas
df_ritmo = procesar_ritmo_play_by_play(RUTA_PBP, RUTA_GAME)
df_four_factors = calcular_four_factors(RUTA_GAME)

# Fusionamos todo
df_partidos_elo1 = generar_features_avanzadas(df_partidos_elo1, df_ritmo, df_four_factors)
df_partidos_elo1, cols_equipos = preparar_datos_ohe(df_partidos_elo1, cols_equipos)

partidos_elo1 = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) <= 2017] 
partidos_elo1_test = df_partidos_elo1[df_partidos_elo1['season_id'].astype(str).str[-4:].astype(int) > 2017] 

todas_cols_escalables = cols_fecha + cols_elo + cols_avanzadas
data_entrada_elo1, data_entrada_elo1_test = escalar_datos(partidos_elo1, partidos_elo1_test, cols_equipos, todas_cols_escalables)
data_salida_elo1, data_salida_elo1_test = preparar_datos_salida(partidos_elo1), preparar_datos_salida(partidos_elo1_test)


Cargando y cruzando bases de datos...
Procesando Ritmo (Pace) desde Play-by-Play...
Calculando los Four Factors de Dean Oliver...


In [9]:
# ----------------- RED NEURONAL -----------------
def crear_modelo_v1_2(n_input):
    modelo = tf.keras.Sequential([
        # Subimos a 256 neuronas en la primera capa porque ahora la red tiene mucha información que procesar
        tf.keras.layers.Dense(256, activation='relu', input_shape=[n_input]),
        tf.keras.layers.Dropout(0.3), 
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2), 
        tf.keras.layers.Dense(2, activation='linear') 
    ])
    
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
        loss=tf.keras.losses.Huber(delta=1.5), 
        metrics=['mean_absolute_error']
    )
    return modelo

def evaluar_precision(modelo, entrada, salida, nombre_modelo):
    predicciones = modelo.predict(entrada, verbose=0)
    mae_home = mean_absolute_error(salida[:, 0], predicciones[:, 0])
    mae_away = mean_absolute_error(salida[:, 1], predicciones[:, 1])
    mae_total = mean_absolute_error(salida, predicciones)
    
    ganador_pred = (predicciones[:, 0] > predicciones[:, 1]).astype(int)
    ganador_real = (salida[:, 0] > salida[:, 1]).astype(int)
    precision = np.mean(ganador_pred == ganador_real) * 100

    print(f"\n--- Resultados {nombre_modelo} ---")
    print(f"Error Promedio Puntos (MAE Total): {mae_total:.2f}")
    print(f"  -> Error Medio Local: {mae_home:.2f}")
    print(f"  -> Error Medio Visitante: {mae_away:.2f}")
    print(f"Precisión Ganador (deducida): {precision:.2f}%")
    return precision, mae_total

modelo_elo1_v1_2 = crear_modelo_v1_2(data_entrada_elo1.shape[1])
print("Entrenando Modelo Definitivo (Pace + Four Factors + B2B) ...")

history = modelo_elo1_v1_2.fit(
    data_entrada_elo1, data_salida_elo1, 
    epochs=2000, 
    verbose=0, 
    validation_split=0.1, 
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)]
)

acc_final, mae_final = evaluar_precision(modelo_elo1_v1_2, data_entrada_elo1_test, data_salida_elo1_test, "Súper Modelo v1.2 Definitivo")

Entrenando Modelo Definitivo (Pace + Four Factors + B2B) ...

--- Resultados Súper Modelo v1.2 Definitivo ---
Error Promedio Puntos (MAE Total): 10.07
  -> Error Medio Local: 10.27
  -> Error Medio Visitante: 9.86
Precisión Ganador (deducida): 65.55%
